# Поиск статей Авито: рабочая leaderboard-база

Pipeline состоит из трёх понятных уровней:

1. TF-IDF и BM25 ищут точные лексические совпадения;
2. локальный `multilingual-e5-small` ищет похожие по смыслу фрагменты;
3. `HistGradientBoosting` сортирует top-50 по content-признакам;
4. query kNN переносит article labels от похожих calibration-запросов;
5. фиксированный RRF `1×retrieval + 2×reranker + 2×query-kNN` формирует ответ.

Это логика решения из коммита `867e551`, давшего около 0.59 на leaderboard. В notebook оставлен один фиксированный pipeline без экспериментальных веток.

## 1. Настройки и данные

Единственная нейронная модель — локальная `intfloat/multilingual-e5-small` (MIT, 117.7M параметров). Зафиксированная ревизия загружается только из локального Hugging Face cache.

In [151]:
from functools import lru_cache
from pathlib import Path
import gc
import os
import random
import re
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
from bs4 import BeautifulSoup
from IPython.display import display
from nltk.stem.snowball import SnowballStemmer
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from transformers import AutoModel, AutoTokenizer

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

DENSE_MODEL_NAME = "intfloat/multilingual-e5-small"
DENSE_MODEL_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"

articles = pd.read_feather(DATA_DIR / "articles.f")
calibration = pd.read_feather(DATA_DIR / "calibration.f")
test = pd.read_feather(DATA_DIR / "test.f")

assert articles.columns.tolist() == ["article_id", "title", "body"]
assert calibration.columns.tolist() == ["query_id", "query_text", "ground_truth"]
assert test.columns.tolist() == ["query_id", "query_text"]
assert not articles["article_id"].duplicated().any()
assert not calibration["query_id"].duplicated().any()
assert not test["query_id"].duplicated().any()
print(f"Статьи: {len(articles)}, calibration: {len(calibration)}, test: {len(test)}")

Статьи: 793, calibration: 500, test: 500


## 2. Очистка и стемминг

HTML превращается в видимые блоки, точные длинные повторы удаляются. Стемминг используется в BM25.

In [152]:
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).lower().replace("ё", "е")).strip()


def html_to_blocks(html):
    soup = BeautifulSoup(str(html), "html.parser")
    for tag in soup(["script", "style", "noscript", "template", "svg"]):
        tag.decompose()
    result, seen = [], set()
    for raw_block in soup.stripped_strings:
        block = normalize_text(raw_block)
        key = re.sub(r"\W+", " ", block).strip()
        if len(key) >= 40 and key in seen:
            continue
        result.append(block)
        if len(key) >= 40:
            seen.add(key)
    return result


TOKEN_RE = re.compile(r"[a-zа-я0-9]+")
stemmer = SnowballStemmer("russian")


@lru_cache(maxsize=100_000)
def stem_token(token):
    return stemmer.stem(token)


def stem_text(text):
    return " ".join(stem_token(token) for token in TOKEN_RE.findall(text))


articles["title_clean"] = articles["title"].map(normalize_text)
articles["body_blocks"] = articles["body"].map(html_to_blocks)
articles["body_clean"] = articles["body_blocks"].str.join(" ")
articles["title_stem"] = articles["title_clean"].map(stem_text)
articles["body_stem"] = articles["body_clean"].map(stem_text)
calibration["query_clean"] = calibration["query_text"].map(normalize_text)
test["query_clean"] = test["query_text"].map(normalize_text)
calibration["query_stem"] = calibration["query_clean"].map(stem_text)
test["query_stem"] = test["query_clean"].map(stem_text)
calibration["relevant_ids"] = calibration["ground_truth"].map(
    lambda value: [int(item) for item in value.split()]
)

duplicate = "достаточно длинный повторяющийся блок для автоматической проверки очистки"
assert html_to_blocks(f"<p>{duplicate}</p><div>{duplicate}</div>") == [duplicate]

## 3. Метрики и пять folds

Каждый calibration-запрос ровно один раз становится validation-примером. Это особенно важно для обучаемого reranker: модель не должна видеть правильные ответы проверяемого fold.

In [153]:
def average_precision_at_k(predicted, relevant, k=10):
    relevant, seen = set(relevant), set()
    hits = 0
    score = 0.0
    for rank, item in enumerate(predicted[:k], 1):
        if item in relevant and item not in seen:
            hits += 1
            score += hits / rank
        seen.add(item)
    return score / min(len(relevant), k) if relevant else 0.0


def recall_at_k(predicted, relevant, k):
    return len(set(predicted[:k]) & set(relevant)) / len(set(relevant))


def rank_articles(scores, k=50):
    order = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    return articles["article_id"].to_numpy()[order]


truth = calibration["relevant_ids"].tolist()

## 4. Lexical retrieval

Сохраняем не только итоговый lexical RRF, но и отдельные scores word TF-IDF, char TF-IDF и BM25. Позже они станут признаками лёгкого reranker.

In [154]:
TITLE_WEIGHT = 0.10


def fit_tfidf(analyzer):
    models, matrices = {}, {}
    for field in ("title", "body"):
        vectorizer = (
            TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                            max_features=180_000, dtype=np.float32)
            if analyzer == "word" else
            TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                            sublinear_tf=True, max_features=200_000, dtype=np.float32)
        )
        matrices[field] = vectorizer.fit_transform(articles[f"{field}_clean"])
        models[field] = vectorizer
    return models, matrices


def tfidf_parts(models, matrices, queries):
    parts = {
        field: (models[field].transform(queries) @ matrices[field].T).toarray()
        for field in ("title", "body")
    }
    parts["combined"] = (
        TITLE_WEIGHT * parts["title"] + (1 - TITLE_WEIGHT) * parts["body"]
    )
    return parts


def fit_bm25(documents, k1=1.5, b=0.75):
    vectorizer = CountVectorizer()
    counts = vectorizer.fit_transform(documents).astype(np.float32)
    lengths = np.asarray(counts.sum(axis=1)).ravel()
    df = np.asarray((counts > 0).sum(axis=0)).ravel()
    idf = np.log1p((len(documents) - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(len(documents)), np.diff(counts.indptr))
    tf = counts.data
    counts.data = (
        tf * (k1 + 1)
        / (tf + k1 * (1 - b + b * lengths[rows] / lengths.mean()))
        * idf[counts.indices]
    )
    return vectorizer, counts


def bm25_scores(vectorizer, matrix, queries):
    query_matrix = vectorizer.transform(queries).astype(bool).astype(np.float32)
    return (query_matrix @ matrix.T).toarray()


def row_normalize(scores):
    return scores / np.maximum(scores.max(axis=1, keepdims=True), 1e-9)


def rrf(score_matrices, weights, k=20):
    fused = np.zeros_like(score_matrices[0], dtype=np.float32)
    rows = np.arange(len(fused))[:, None]
    for weight, scores in zip(weights, score_matrices):
        order = np.argsort(-scores, axis=1, kind="stable")
        ranks = np.empty_like(order)
        ranks[rows, order] = np.arange(1, scores.shape[1] + 1)
        fused += weight / (k + ranks)
    return fused


word_models, word_matrices = fit_tfidf("word")
char_models, char_matrices = fit_tfidf("char")
bm25_models, bm25_matrices = {}, {}
for field in ("title", "body"):
    bm25_models[field], bm25_matrices[field] = fit_bm25(articles[f"{field}_stem"])


def lexical_parts(clean_queries, stem_queries):
    word = tfidf_parts(word_models, word_matrices, clean_queries)
    char = tfidf_parts(char_models, char_matrices, clean_queries)
    bm25 = {
        field: row_normalize(
            bm25_scores(bm25_models[field], bm25_matrices[field], stem_queries)
        )
        for field in ("title", "body")
    }
    bm25["combined"] = (
        TITLE_WEIGHT * bm25["title"] + (1 - TITLE_WEIGHT) * bm25["body"]
    )
    lexical = rrf(
        [word["combined"], char["combined"], bm25["combined"]], [1, 1, 2]
    )
    return {"word": word, "char": char, "bm25": bm25, "lexical": lexical}


cal_lexical = lexical_parts(calibration["query_clean"], calibration["query_stem"])
test_lexical = lexical_parts(test["query_clean"], test["query_stem"])

## 5. Dense retrieval и агрегация chunks

E5 кодирует chunks с префиксом `passage:`, запросы — с `query:`. Для каждой статьи сохраняются два лучших chunk-score: это позволяет проверить, полезен ли второй релевантный фрагмент.

In [155]:
dense_tokenizer = AutoTokenizer.from_pretrained(
    DENSE_MODEL_NAME, revision=DENSE_MODEL_REVISION, local_files_only=True
)
dense_model = AutoModel.from_pretrained(
    DENSE_MODEL_NAME, revision=DENSE_MODEL_REVISION, local_files_only=True
).eval()


def article_chunks(title, blocks, chunk_tokens=192, overlap=24):
    title_ids = dense_tokenizer.encode(title, add_special_tokens=False)[:48]
    body_ids = []
    for block in blocks:
        body_ids.extend(dense_tokenizer.encode(block, add_special_tokens=False))
        body_ids.append(dense_tokenizer.sep_token_id)
    payload = chunk_tokens - len(title_ids) - 1
    step = max(1, payload - overlap)
    return [
        dense_tokenizer.decode(title_ids + [dense_tokenizer.sep_token_id] + body_ids[start:start + payload])
        for start in range(0, max(1, len(body_ids)), step)
    ]


chunks, chunk_article_indices = [], []
for article_index, row in articles.iterrows():
    parts = article_chunks(row["title_clean"], row["body_blocks"])
    chunks.extend(parts)
    chunk_article_indices.extend([article_index] * len(parts))
chunk_article_indices = np.asarray(chunk_article_indices)


def mean_pool(hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1)
    return (hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)


def embed(texts, prefix, batch_size=32):
    result = []
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            prepared = [prefix + text for text in texts[start:start + batch_size]]
            batch = dense_tokenizer(
                prepared, padding=True, truncation=True, max_length=192, return_tensors="pt"
            )
            output = dense_model(**batch).last_hidden_state
            vectors = mean_pool(output, batch["attention_mask"])
            result.append(torch.nn.functional.normalize(vectors).cpu().numpy())
    return np.vstack(result)


chunk_embeddings = embed(chunks, "passage: ")


def dense_article_scores(query_embeddings):
    similarities = query_embeddings @ chunk_embeddings.T
    scores = np.empty((len(query_embeddings), len(articles)), dtype=np.float32)
    second_scores = np.empty_like(scores)
    best_chunks = np.empty((len(query_embeddings), len(articles)), dtype=np.int32)
    for article_index in range(len(articles)):
        positions = np.flatnonzero(chunk_article_indices == article_index)
        local_scores = similarities[:, positions]
        local_best = local_scores.argmax(axis=1)
        scores[:, article_index] = local_scores[np.arange(len(query_embeddings)), local_best]
        if len(positions) > 1:
            second_scores[:, article_index] = np.partition(local_scores, -2, axis=1)[:, -2]
        else:
            second_scores[:, article_index] = scores[:, article_index]
        best_chunks[:, article_index] = positions[local_best]
    return scores, second_scores, best_chunks


calibration_query_embeddings = embed(list(calibration["query_clean"]), "query: ")
test_query_embeddings = embed(list(test["query_clean"]), "query: ")
dense_calibration, dense_second_calibration, best_chunk_calibration = dense_article_scores(
    calibration_query_embeddings
)
dense_test, dense_second_test, best_chunk_test = dense_article_scores(test_query_embeddings)

semantic_groups = AgglomerativeClustering(
    n_clusters=None, distance_threshold=0.15, metric="cosine", linkage="complete"
).fit_predict(calibration_query_embeddings)
folds = list(StratifiedGroupKFold(
    5, shuffle=True, random_state=RANDOM_STATE
).split(
    calibration_query_embeddings,
    calibration["relevant_ids"].str.len().clip(upper=3),
    semantic_groups,
))
assert sorted(np.concatenate([valid for _, valid in folds]).tolist()) == list(range(len(calibration)))
assert all(
    set(semantic_groups[train]).isdisjoint(semantic_groups[valid])
    for train, valid in folds
)

print(
    f"Фрагментов: {len(chunks)}, embedding shape: {chunk_embeddings.shape}, "
    f"параметров E5: {sum(p.numel() for p in dense_model.parameters()) / 1e6:.1f}M, "
    f"semantic groups: {len(np.unique(semantic_groups))}"
)
del dense_model, chunk_embeddings
gc.collect()

Фрагментов: 6967, embedding shape: (6967, 384), параметров E5: 117.7M, semantic groups: 81


24846

## 6. Фиксированный retrieval

Используется конфигурация рабочего leaderboard-решения: `RRF(2×lexical + dense top-2)`.

In [156]:
def map10(scores):
    predictions = rank_articles(scores, 10)
    return np.mean([
        average_precision_at_k(predictions[index], truth[index])
        for index in range(len(truth))
    ])


dense_top2_calibration = 0.8 * dense_calibration + 0.2 * dense_second_calibration
dense_top2_test = 0.8 * dense_test + 0.2 * dense_second_test

RETRIEVAL_NAME = "RRF 2×lexical + dense top-2"
retrieval_calibration = rrf(
    [cal_lexical["lexical"], dense_top2_calibration], [2, 1]
)
retrieval_test = rrf(
    [test_lexical["lexical"], dense_top2_test], [2, 1]
)
print(f"{RETRIEVAL_NAME}: calibration MAP@10={map10(retrieval_calibration):.6f}")

RRF 2×lexical + dense top-2: calibration MAP@10=0.415933


## 7. Признаки, reranker и query kNN

Для top-50 retrieval-кандидатов используются только признаки финальной
модели: scores и ranks word/char TF-IDF, BM25, title, E5 и общего retrieval,
а также нормализация внутри запроса и простые token-overlap признаки.

`HistGradientBoosting` оценивается out-of-fold. Query kNN переносит метки
только от train-запросов соответствующего fold.


In [157]:
def reciprocal_rank_features(scores):
    order = np.argsort(-scores, axis=1, kind="stable")
    ranks = np.empty_like(order)
    ranks[
        np.arange(len(scores))[:, None], order
    ] = np.arange(1, scores.shape[1] + 1)
    return 1 / (20 + ranks)


def feature_cube(parts, dense, dense_second, retrieval):
    score_features = [
        parts["word"]["combined"],
        parts["word"]["title"],
        parts["char"]["combined"],
        parts["char"]["title"],
        parts["bm25"]["combined"],
        parts["bm25"]["title"],
        parts["lexical"],
        dense,
        dense_second,
        retrieval,
    ]
    rank_features = [
        reciprocal_rank_features(scores)
        for scores in score_features
    ]
    return np.stack(
        [*score_features, *rank_features], axis=2
    ).astype(np.float32)


title_sets = [
    set(text.split()) for text in articles["title_stem"]
]
body_sets = [
    set(text.split()) for text in articles["body_stem"]
]


def enrich_features(
    base, parts, dense, dense_second, retrieval, query_stems
):
    score_features = [
        parts["word"]["combined"],
        parts["word"]["title"],
        parts["char"]["combined"],
        parts["char"]["title"],
        parts["bm25"]["combined"],
        parts["bm25"]["title"],
        parts["lexical"],
        dense,
        dense_second,
        retrieval,
    ]
    z_scores = [
        (score - score.mean(1, keepdims=True))
        / np.maximum(score.std(1, keepdims=True), 1e-6)
        for score in score_features
    ]
    gaps = [
        score.max(1, keepdims=True) - score
        for score in score_features
    ]
    overlap = np.zeros(
        (len(query_stems), len(articles), 7), dtype=np.float32
    )
    for query_index, query in enumerate(query_stems):
        query_set = set(query.split())
        denominator = max(1, len(query_set))
        for article_index, (title, body) in enumerate(
            zip(title_sets, body_sets)
        ):
            title_match = len(query_set & title)
            body_match = len(query_set & body)
            overlap[query_index, article_index] = (
                title_match / denominator,
                body_match / denominator,
                title_match / max(1, len(query_set | title)),
                body_match / max(1, len(query_set | body)),
                np.log1p(len(title)),
                np.log1p(len(body)),
                np.log1p(len(query_set)),
            )
    return np.concatenate(
        [
            base,
            np.stack(z_scores, axis=2),
            np.stack(gaps, axis=2),
            overlap,
        ],
        axis=2,
    ).astype(np.float32)


calibration_features = enrich_features(
    feature_cube(
        cal_lexical,
        dense_calibration,
        dense_second_calibration,
        retrieval_calibration,
    ),
    cal_lexical,
    dense_calibration,
    dense_second_calibration,
    retrieval_calibration,
    calibration["query_stem"],
)
test_features = enrich_features(
    feature_cube(
        test_lexical,
        dense_test,
        dense_second_test,
        retrieval_test,
    ),
    test_lexical,
    dense_test,
    dense_second_test,
    retrieval_test,
    test["query_stem"],
)
candidate_indices = np.argsort(
    -retrieval_calibration, axis=1, kind="stable"
)[:, :50]
test_candidate_indices = np.argsort(
    -retrieval_test, axis=1, kind="stable"
)[:, :50]

article_ids = articles["article_id"].to_numpy()
labels = np.zeros(
    (len(calibration), len(articles)), dtype=np.int8
)
for query_index, relevant in enumerate(truth):
    labels[query_index] = np.isin(article_ids, relevant)


def candidate_rows(features, candidates, query_indices):
    return np.vstack([
        features[index, candidates[index]]
        for index in query_indices
    ])


def candidate_labels(candidates, query_indices):
    return np.concatenate([
        labels[index, candidates[index]]
        for index in query_indices
    ])


def new_gradient_boosting():
    return HistGradientBoostingClassifier(
        learning_rate=0.03,
        max_iter=200,
        max_leaf_nodes=15,
        min_samples_leaf=10,
        l2_regularization=2.0,
        random_state=RANDOM_STATE,
    )


gradient_oof = np.full(
    (len(calibration), len(articles)),
    -1e9,
    dtype=np.float32,
)
for train_indices, validation_indices in folds:
    model = new_gradient_boosting()
    x_train = candidate_rows(
        calibration_features, candidate_indices, train_indices
    )
    y_train = candidate_labels(
        candidate_indices, train_indices
    )
    model.fit(
        x_train,
        y_train,
        sample_weight=compute_sample_weight("balanced", y_train),
    )
    for query_index in validation_indices:
        candidates = candidate_indices[query_index]
        gradient_oof[query_index, candidates] = (
            model.predict_proba(
                calibration_features[query_index, candidates]
            )[:, 1]
        )

content_oof = rrf(
    [retrieval_calibration, gradient_oof], [1, 2]
)


query_word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True
)
query_char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=2,
    sublinear_tf=True,
)
calibration_query_word = query_word_vectorizer.fit_transform(
    calibration["query_clean"]
)
test_query_word = query_word_vectorizer.transform(
    test["query_clean"]
)
calibration_query_char = query_char_vectorizer.fit_transform(
    calibration["query_clean"]
)
test_query_char = query_char_vectorizer.transform(
    test["query_clean"]
)
calibration_query_similarity = (
    0.4
    * (
        calibration_query_word
        @ calibration_query_word.T
    ).toarray()
    + 0.6
    * (
        calibration_query_char
        @ calibration_query_char.T
    ).toarray()
)
test_query_similarity = (
    0.4
    * (
        test_query_word @ calibration_query_word.T
    ).toarray()
    + 0.6
    * (
        test_query_char @ calibration_query_char.T
    ).toarray()
)
np.fill_diagonal(calibration_query_similarity, -1)


def transfer_query_labels(
    similarity, reference_indices, neighbors=20, power=2
):
    nearest = np.argsort(
        -similarity, axis=1, kind="stable"
    )[:, :neighbors]
    nearest_similarity = np.take_along_axis(
        similarity, nearest, axis=1
    ).clip(min=0)
    neighbor_labels = labels[reference_indices][nearest]
    return np.einsum(
        "ij,ijk->ik",
        nearest_similarity ** power,
        neighbor_labels,
    )


knn_oof = np.zeros_like(labels, dtype=np.float32)
for train_indices, validation_indices in folds:
    knn_oof[validation_indices] = transfer_query_labels(
        calibration_query_similarity[
            np.ix_(validation_indices, train_indices)
        ],
        train_indices,
    )
knn_test = transfer_query_labels(
    test_query_similarity, np.arange(len(calibration))
)
final_oof = rrf(
    [retrieval_calibration, gradient_oof, knn_oof],
    [1, 2, 2],
)

## 8. Фиксированная grouped-validation

Здесь нет выбора весов или экспериментальных веток. Таблица показывает
последовательный вклад компонентов единственного финального pipeline.


In [158]:
def evaluate(name, scores):
    predictions = rank_articles(scores, 50)
    fold_map = [
        np.mean([
            average_precision_at_k(
                predictions[index], truth[index]
            )
            for index in validation
        ])
        for _, validation in folds
    ]
    return {
        "method": name,
        **{
            f"fold {number}": value
            for number, value in enumerate(fold_map, 1)
        },
        "MAP@10 mean": np.mean(fold_map),
        "MAP@10 std": np.std(fold_map),
        **{
            f"Recall@{k}": np.mean([
                recall_at_k(
                    predictions[index], truth[index], k
                )
                for index in range(len(truth))
            ])
            for k in (10, 20, 50)
        },
    }


comparison = pd.DataFrame([
    evaluate("Lexical", cal_lexical["lexical"]),
    evaluate("Dense E5-small", dense_top2_calibration),
    evaluate(RETRIEVAL_NAME, retrieval_calibration),
    evaluate("HistGB content-only", content_oof),
    evaluate("Final HistGB + query kNN", final_oof),
])
numeric_columns = comparison.columns.drop("method")
display(comparison.style.format({
    column: "{:.6f}" for column in numeric_columns
}))

final_row = comparison.iloc[-1]
FINAL_MAP10 = float(final_row["MAP@10 mean"])
print(
    f"Зафиксировано: RRF(1×retrieval + 2×HistGB "
    f"+ 2×query-kNN); 5-fold MAP@10={FINAL_MAP10:.6f}"
)

,method,fold 1,fold 2,fold 3,fold 4,fold 5,MAP@10 mean,MAP@10 std,Recall@10,Recall@20,Recall@50
0,Lexical,0.289213,0.397346,0.401208,0.279054,0.307408,0.334846,0.053401,0.652000,0.801167,0.930000
1,Dense E5-small,0.288741,0.454101,0.373653,0.315844,0.274632,0.341394,0.065766,0.648667,0.780500,0.917000
2,RRF 2×lexical + dense top-2,0.370875,0.500919,0.448005,0.383281,0.425172,0.425651,0.046820,0.758333,0.865167,0.961000
3,HistGB content-only,0.428491,0.630212,0.556807,0.478161,0.524826,0.523700,0.068674,0.831500,0.931833,0.961000
4,Final HistGB + query kNN,0.511341,0.677697,0.606217,0.613168,0.657189,0.613123,0.057478,0.921333,0.957667,0.980000


Зафиксировано: RRF(1×retrieval + 2×HistGB + 2×query-kNN); 5-fold MAP@10=0.613123


## 9. Финальное обучение и `answer.csv`

HistGB обучается один раз на всех calibration-кандидатах. Затем фиксированный
RRF формирует top-10 и выполняются проверки формата submission.


In [159]:
all_indices = np.arange(len(calibration))
x_all = candidate_rows(
    calibration_features, candidate_indices, all_indices
)
y_all = candidate_labels(candidate_indices, all_indices)

final_model = new_gradient_boosting()
final_model.fit(
    x_all,
    y_all,
    sample_weight=compute_sample_weight("balanced", y_all),
)
test_learned = np.full(
    (len(test), len(articles)), -1e9, dtype=np.float32
)
for query_index, candidates in enumerate(
    test_candidate_indices
):
    test_learned[query_index, candidates] = (
        final_model.predict_proba(
            test_features[query_index, candidates]
        )[:, 1]
    )

final_scores = rrf(
    [retrieval_test, test_learned, knn_test], [1, 2, 2]
)
test_predictions = rank_articles(final_scores, 10)
answer = test[["query_id"]].copy()
answer["answer"] = [
    " ".join(map(str, row)) for row in test_predictions
]
answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(answer_path, index=False)

saved = pd.read_csv(
    answer_path, dtype={"answer": "string"}
)
parsed = saved["answer"].str.split().map(
    lambda values: list(map(int, values))
)
known_ids = set(articles["article_id"])
assert len(saved) == len(test)
assert saved.columns.tolist() == ["query_id", "answer"]
assert not saved.isna().any().any()
assert parsed.map(len).eq(10).all()
assert parsed.map(
    lambda values: len(values) == len(set(values))
).all()
assert parsed.map(
    lambda values: set(values) <= known_ids
).all()
assert (
    saved["query_id"].tolist()
    == test["query_id"].tolist()
)

display(saved.head())
print(
    f"Сохранено: {answer_path}; строк: {len(saved)}; "
    f"5-fold MAP@10: {FINAL_MAP10:.6f}"
)

,query_id,answer
0,1,4219 4396 4234 4308 4384 1951 4361 1909 4331 1899
1,2,4219 4384 2646 4308 1923 4009 4400 2408 4387 2802
2,3,1909 4234 4396 4400 4219 4286 4308 4361 4329 1907
3,4,4219 4234 2646 4400 2865 1966 1951 4214 4384 2698
4,5,4214 2698 4308 4234 4219 4384 1747 2948 4388 3128


Сохранено: output/answer.csv; строк: 500; 5-fold MAP@10: 0.613123
